# COSC 2060: Modern Data Engineering & AI Infrastructure
## Unit 02: Re-Platforming High-Throughput Anomaly Detection (Pandas `groupby.apply` → Polars Lazy Expressions)

---

### 🏢 The Workplace Scenario
Your team's lead data scientist, **Dr. Marcus Vance**, created a fleet anomaly detection algorithm to flag sudden fuel consumption spikes and engine thermal spikes across 500 commercial transport trucks.

In his exploratory Jupyter Notebook, Dr. Vance partitioned the fleet by `vehicle_id` and used Pandas `df.groupby('vehicle_id').apply(...)` with custom rolling window calculations. While this worked fine on his local 1,000-row sample, the data engineering cron job choked when processing the hourly batch of **1,000,000 streaming telemetry events**. The Python Global Interpreter Lock (GIL) pegged a single CPU core at 100%, memory usage skyrocketed to 2.4 GB, and the job breached its SLA (Service Level Agreement).

### 🎯 Your Mission as AI Data Engineer
1. **Analyze Dr. Vance's Prototype**: Understand the rolling z-score logic used to isolate anomalous fuel drops ($|z| > 3.0$) partitioned per vehicle.
2. **Expose the Python GIL Bottleneck**: Measure how Pandas `groupby.apply` forces Python object overhead, single-threaded execution, and repetitive DataFrame copies.
3. **Re-Platform into Polars**: Refactor the pipeline into **Polars LazyFrame expressions** using native multi-threaded Apache Arrow columnar memory and the `.over('vehicle_id')` window syntax.
4. **Assert Mathematical Parity**: Prove that the anomaly flags and z-scores generated by Polars match Dr. Vance's prototype with 100% precision.
5. **Benchmark the Multi-Threaded Speedup**: Measure execution time and demonstrate sub-second throughput on 1,000,000 records.

In [ ]:
import numpy as np
import pandas as pd
import polars as pl
import time

# Set deterministic seed
np.random.seed(42)

print("[*] Generating 1,000-row sample telemetry dataset for Dr. Vance's prototype...")

n_sample = 1000
vehicles = [f"VIN-{i:03d}" for i in range(10)]

sample_telemetry = pd.DataFrame({
    "event_id": [f"EVT-{i:06d}" for i in range(n_sample)],
    "vehicle_id": np.random.choice(vehicles, size=n_sample),
    "timestamp_sec": np.sort(np.random.randint(0, 3600, size=n_sample)),
    "fuel_rate_gph": np.round(np.random.normal(8.5, 1.2, size=n_sample), 2),
    "engine_temp_c": np.round(np.random.normal(88.0, 4.0, size=n_sample), 1)
}).sort_values(["vehicle_id", "timestamp_sec"]).reset_index(drop=True)

# Inject known anomaly spikes into the sample for verification
sample_telemetry.loc[45, "fuel_rate_gph"] = 18.5  # extreme fuel spike
sample_telemetry.loc[120, "engine_temp_c"] = 115.0 # overheating spike

print(f"[+] Sample Data Shape: {sample_telemetry.shape}")
sample_telemetry.head(4)

In [ ]:
# ==============================================================================
# TIER 1: The Data Science Prototype (Dr. Marcus Vance)
# ==============================================================================
# Dr. Vance computes a 20-sample rolling mean and rolling std for each vehicle,
# calculates the fuel z-score, and flags any reading with |z| > 3.0 as an anomaly.

t0 = time.time()

def detect_anomalies_pandas(group):
    roll = group["fuel_rate_gph"].rolling(window=20, min_periods=1)
    mean_val = roll.mean()
    std_val = roll.std().fillna(1.0)
    
    group["fuel_rolling_mean"] = mean_val
    group["fuel_rolling_std"] = std_val
    group["fuel_z_score"] = (group["fuel_rate_gph"] - mean_val) / std_val
    group["is_fuel_anomaly"] = group["fuel_z_score"].abs() > 3.0
    return group

ds_prototype_result = sample_telemetry.groupby("vehicle_id", group_keys=False).apply(
    detect_anomalies_pandas
).reset_index(drop=True)

prototype_duration = time.time() - t0
print(f"[+] Prototype executed in {prototype_duration:.4f} seconds.")

flagged = ds_prototype_result[ds_prototype_result["is_fuel_anomaly"]]
print(f"[+] Anomalies Detected: {len(flagged)}")
flagged[["event_id", "vehicle_id", "fuel_rate_gph", "fuel_z_score", "is_fuel_anomaly"]]

### ⚠️ The Scaling Bottleneck: Why `groupby.apply()` Collapses

When scaling Dr. Vance's code to production, three critical performance flaws emerge:
1. **The Python Interpreter Loop**: In Pandas, `.apply()` converts each grouped chunk back into a separate Python object, executes pure Python function calls line-by-line, and reassembles the pieces. The Python Global Interpreter Lock (GIL) restricts this entire process to a single CPU core.
2. **Intermediate Memory Allocations**: For 500 vehicles across 1,000,000 rows, Pandas creates 500 separate temporary DataFrame slices, driving memory fragmentation.
3. **No Query Optimization**: Pandas executes operations eagerly. It cannot prune unused columns or fuse the calculation of `mean` and `std` into a single vectorized pass.

Let's test this by scaling to **600,000 telemetry records** across 100 vehicles.

In [ ]:
# ==============================================================================
# TIER 2: The Scale Stress-Test (600,000 records)
# ==============================================================================
n_scale = 600000
scale_vehicles = [f"VIN-{i:03d}" for i in range(100)]

print(f"[*] Generating scaled telemetry dataset ({n_scale:,} rows across 100 vehicles)...")

df_scale_pandas = pd.DataFrame({
    "event_id": [f"EVT-{i:07d}" for i in range(n_scale)],
    "vehicle_id": np.random.choice(scale_vehicles, size=n_scale),
    "timestamp_sec": np.sort(np.random.randint(0, 86400 * 3, size=n_scale)),
    "fuel_rate_gph": np.round(np.random.normal(8.5, 1.2, size=n_scale), 2),
    "engine_temp_c": np.round(np.random.normal(88.0, 4.0, size=n_scale), 1)
}).sort_values(["vehicle_id", "timestamp_sec"]).reset_index(drop=True)

# Inject known anomaly spikes
df_scale_pandas.loc[1000, "fuel_rate_gph"] = 22.0
df_scale_pandas.loc[50000, "fuel_rate_gph"] = 28.5

print("[*] Timing Pandas groupby.apply on 600,000 rows...")
t_pandas_scale_start = time.time()

pandas_scale_result = df_scale_pandas.groupby("vehicle_id", group_keys=False).apply(
    detect_anomalies_pandas
).reset_index(drop=True)

pandas_scale_duration = time.time() - t_pandas_scale_start
print(f"[!] Pandas Scale Duration: {pandas_scale_duration:.3f} seconds")

### 🚀 The Data Engineering Solution: Polars Lazy Expressions

To deliver production-grade performance, we re-platform into **Polars**:
1. **Rust Columnar Core**: Polars is built from scratch in Rust on top of the Apache Arrow columnar format, enabling zero-copy SIMD parallel execution.
2. **Multi-Threaded Window Expressions (`.over`)**: Instead of splitting data into hundreds of Python DataFrame objects, Polars processes windows in-place in parallel across all CPU cores using `pl.col('fuel_rate_gph').rolling_mean(...).over('vehicle_id')`.
3. **Lazy Query Optimizer (`.lazy()`)**: Polars builds a logical query plan, optimizes column projections, and streams execution through cache-conscious memory blocks.

In [ ]:
# ==============================================================================
# TIER 3: The Data Engineering Refactor (Polars Lazy Expressions)
# ==============================================================================
print("[*] Executing Polars multi-threaded lazy query plan...")

# Convert Pandas DataFrame to Polars DataFrame (Arrow-backed)
polars_input = pl.from_pandas(df_scale_pandas)

t_polars_start = time.time()

polars_query = (
    polars_input.lazy()
    .with_columns([
        pl.col("fuel_rate_gph")
          .rolling_mean(window_size=20, min_samples=1)
          .over("vehicle_id")
          .alias("fuel_rolling_mean"),
        pl.col("fuel_rate_gph")
          .rolling_std(window_size=20, min_samples=1)
          .over("vehicle_id")
          .fill_null(1.0)
          .alias("fuel_rolling_std")
    ])
    .with_columns([
        ((pl.col("fuel_rate_gph") - pl.col("fuel_rolling_mean")) / pl.col("fuel_rolling_std"))
          .alias("fuel_z_score")
    ])
    .with_columns([
        (pl.col("fuel_z_score").abs() > 3.0).alias("is_fuel_anomaly")
    ])
)

# Collect executes the optimized query plan across all available CPU threads
polars_scale_result = polars_query.collect()
polars_scale_duration = time.time() - t_polars_start

print(f"[+] Polars Execution Duration: {polars_scale_duration:.3f} seconds")
print(f"[+] Output Shape: {polars_scale_result.shape}")
polars_scale_result.head(4)

In [ ]:
# ==============================================================================
# QUALITY GATE: Mathematical Equivalence Assertion
# ==============================================================================
# Verify that the Polars Rust engine generated identical anomaly classifications
# and z-scores to Dr. Vance's Python prototype.

print("[*] Validating mathematical parity between Pandas and Polars outputs...")

# Convert Polars columns to NumPy for comparison
p_z = pandas_scale_result["fuel_z_score"].to_numpy()
pl_z = polars_scale_result["fuel_z_score"].to_numpy()

p_anom = pandas_scale_result["is_fuel_anomaly"].to_numpy()
pl_anom = polars_scale_result["is_fuel_anomaly"].to_numpy()

# 1. Assert anomaly flag array parity
assert np.array_equal(p_anom, pl_anom), "Anomaly boolean flags do not match!"

# 2. Assert z-score continuous numerical values within 1e-4 tolerance
np.testing.assert_allclose(p_z, pl_z, atol=1e-4, err_msg="Z-scores differ from prototype!")

# 3. Assert total anomaly count match
p_count = int(p_anom.sum())
pl_count = int(pl_anom.sum())
assert p_count == pl_count, f"Anomaly count mismatch: {p_count} vs {pl_count}"

print("=" * 60)
print(f"[+] PASS: 100% MATHEMATICAL PARITY CONFIRMED!")
print(f"    Total Anomalies Detected: {pl_count:,} out of {n_scale:,} records.")
print(f"    Zero false positives or missed anomalies detected.")
print("=" * 60)

In [ ]:
# ==============================================================================
# PERFORMANCE AUDIT: Benchmark Report
# ==============================================================================
speedup = pandas_scale_duration / max(polars_scale_duration, 0.001)

print("-" * 60)
print(f"{'Engine & Architecture':<32} | {'Rows Processed':<15} | {'Duration (s)':<12}")
print("-" * 60)
print(f"{'Pandas (groupby.apply)':<32} | {n_scale:<15,} | {pandas_scale_duration:<12.3f}")
print(f"{'Polars (Lazy Window .over)':<32} | {n_scale:<15,} | {polars_scale_duration:<12.3f}")
print("-" * 60)
print(f"[>>>] SPEEDUP FACTOR: {speedup:.1f}x Faster Execution with Polars")
print("-" * 60)

assert polars_scale_duration < pandas_scale_duration, "Polars should execute faster than Pandas apply!"
print("[+] Performance Benchmark: PASSED.")

### 🎓 Key Takeaways for AI Data Engineers

1. **Avoid `.apply()` in Production**: Pandas `.apply()` is essentially an un-vectorized Python `for` loop wrapped in syntactic sugar. It cannot be multi-threaded because of Python's GIL.
2. **The Power of Window Functions (`.over()`)**: In Polars, window operations like `.over('vehicle_id')` partition data at the native C/Rust level without creating thousands of intermediate Python objects.
3. **Lazy Evaluation Optimizes Resources**: Using `.lazy()` allows Polars to inspect the entire transformation pipeline before executing, fusing operations and allocating memory efficiently.
4. **Bridge to Unit 02 Applied Lab**: In your upcoming lab assignment, you will deploy this Polars engine to process 1,000,000 real-world fuel and vibration sensor records with streaming window aggregations!